In [27]:
import pandas as pd

movies = pd.read_csv('data/movies.csv')

print("Shape:", movies.shape)
print("\nColumn types and non-null counts:")
movies.info()

Shape: (3201, 16)

Column types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Title                   3200 non-null   str    
 1   US Gross                3194 non-null   float64
 2   Worldwide Gross         3194 non-null   float64
 3   US DVD Sales            564 non-null    float64
 4   Production Budget       3200 non-null   float64
 5   Release Date            3201 non-null   str    
 6   MPAA Rating             2596 non-null   str    
 7   Running Time min        1209 non-null   float64
 8   Distributor             2969 non-null   str    
 9   Source                  2836 non-null   str    
 10  Major Genre             2926 non-null   str    
 11  Creative Type           2755 non-null   str    
 12  Director                1870 non-null   str    
 13  Rotten Tomatoes Rating  2321 non-null   float64
 14

What one row is and which columns you will use?

The dataset has 3,201 rows and 16 columns. Each row represents one film.
I will focus on Title, Release Date, Production Budget, Worldwide Gross, Major Genre, Rotten Tomatoes Rating and IMDB Rating

The first problem is Release Date format - it is a string, but should be a datetime. Also check min and max dates

In [28]:
# Convert Release Date to datetime and check min/max dates
movies['Release Date'] = pd.to_datetime(movies['Release Date'])

print("Min date:", movies['Release Date'].min())
print("Max date:", movies['Release Date'].max())

Min date: 1928-12-31 00:00:00
Max date: 2046-12-31 00:00:00


The maximum date is 2046-12-31 is impossible. Checking the repository on GitHub shows the data was not updated after 2020, so any date after 2020 must be an error.

In [29]:
# Check which films have dates in the future (after 2020 for our data)
future_dates = movies['Release Date'].dt.year > 2020
print("Films with future dates:", future_dates.sum())

movies.loc[future_dates, ['Title', 'Release Date']]

Films with future dates: 18


,Title,Release Date
9,Duel in the Sun,2046-12-31
16,Wilson,2044-08-01
33,42nd Street,2033-03-09
85,Ben-Hur,2025-12-30
90,The Best Years of Our Lives,2046-11-21
102,The Big Parade,2025-01-01
174,"Charge of the Light Brigade, The",2036-10-20
221,Cat People,2042-11-16
337,Fantasia,2040-11-13
382,A Guy Named Joe,2043-12-24


18 rows have release dates between 2025 and 2046, which is impossible since the dataset ends in 2020. Instead of guessing the exact dates, I replace them with missing values (NaT) so they do not distort any time-based checks.

In [30]:
movies.loc[future_dates, 'Release Date'] = pd.NaT

print("Min date after fix:", movies['Release Date'].min())
print("Max date after fix:", movies['Release Date'].max())

Min date after fix: 1928-12-31 00:00:00
Max date after fix: 2020-09-17 00:00:00


Check Production Budget

In [31]:
# Check summary statistics, missing values, and zeros in Production Budget
print(movies['Production Budget'].describe())
print("\nMissing values:", movies['Production Budget'].isna().sum())
print("Zero values:", (movies['Production Budget'] == 0).sum())
print("Negative values:", (movies['Production Budget'] < 0).sum())

count    3.200000e+03
mean     3.106917e+07
std      3.558591e+07
min      2.180000e+02
25%      6.575000e+06
50%      2.000000e+07
75%      4.200000e+07
max      3.000000e+08
Name: Production Budget, dtype: float64

Missing values: 1
Zero values: 0
Negative values: 0


In [32]:
# Check the film with the minimum Production Budget
min_budget = movies['Production Budget'].min()
movies.loc[movies['Production Budget'] == min_budget, ['Title', 'Release Date', 'Production Budget', 'Worldwide Gross']]

# Check the film with the maximum Production Budget
max_budget = movies['Production Budget'].max()
movies.loc[movies['Production Budget'] == max_budget, ['Title', 'Release Date', 'Production Budget', 'Worldwide Gross']]

,Title,Release Date,Production Budget,Worldwide Gross
2508,Pirates of the Caribbean: At World's End,2007-05-25,300000000.0,960996492.0


Production Budget has 3200 valid values (1 missing) and no zeros or negative numbers.
The minimum budget ($218 for Tarnation) is a real micro-budget documentary, not an error.
The maximum budget ($300,000,000 for Pirates of the Caribbean: At World's End) is a real budget also.

Therefore, Production Budget needs no cleaning.

Check Worldwide Gross

In [33]:
# Find films with 0 in Worldwide Gross
print("Zero Worldwide Gross:", (movies['Worldwide Gross'] == 0).sum())

movies.loc[movies['Worldwide Gross'] == 0, ['Title', 'Release Date', 'Production Budget', 'Worldwide Gross']].head(10)

Zero Worldwide Gross: 47


,Title,Release Date,Production Budget,Worldwide Gross
19,12 Angry Men,1957-04-13,340000.0,0.0
21,1776,1972-11-09,4000000.0,0.0
48,Alice in Wonderland,1951-07-28,3000000.0,0.0
68,Barbarella,1968-10-10,9000000.0,0.0
94,Big Things,2009-12-31,50000.0,0.0
123,The Blood of My Brother: A Story of Death in Iraq,2006-06-30,120000.0,0.0
147,Batman - The Movie,2001-08-21,1377800.0,0.0
162,Crop Circles: Quest for Truth,2002-08-23,600000.0,0.0
174,"Charge of the Light Brigade, The",NaT,1200000.0,0.0
211,Crying With Laughter,2009-12-31,820000.0,0.0


47 films have 0 in Worldwide Gross, even though they have real production budgets. Here 0 means missing data and leaving 0 would pull down average earnings.

In [25]:
# Replace 0 with missing values
movies['Worldwide Gross'] = movies['Worldwide Gross'].replace(0, pd.NA)

print("Zero Worldwide Gross after fix:", (movies['Worldwide Gross'] == 0).sum())

Zero Worldwide Gross after fix: 0
